# Hotpot phrase/token embedding collection experiment

这个 notebook 复用 `build_hotpot_retrieval_dataset` 和 `extract_important_spans` 的数据流，
但不构图、不建节点，只收集每个 phrase/token 在数据库中所有出现位置对应的 embedding。

最终得到的数据结构 `embedding_store` 支持：
- 输入 phrase 或 token，取回它的全部 embedding 记录
- 通过 `document_idx` 直接回查 `documents` 里的原始文本
- 通过 `cleaned_documents` 回查用于 span 对齐的清洗后文本

模型路径和 spaCy 设定保持与 `ProtoGraphRAG` 一致。

In [ ]:
from collections import defaultdict
from pathlib import Path
import pickle

import numpy as np
import spacy
import torch
from spacy.tokenizer import Tokenizer
from spacy.util import compile_infix_regex
from transformers import AutoModel, AutoTokenizer

from text_processing import (
    clean_text,
    encode_chunk_batch,
    extract_important_spans,
    get_token_indices_for_phrase,
    normalize_text,
)
from utils import build_hotpot_retrieval_dataset

In [ ]:
file_path = "hotpot_dev_distractor_v1.json"
num_samples = 500

spacy_model_name = "en_core_web_lg"
text_encoder_name_or_path = "/home/xiaoyue/ProtoGraphRAG/deberta-v3-large"

device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 8
remove_duplicate_token = True
discard_no_word = False

output_path = Path("hotpot_phrase_token_embedding_index.pkl")

In [ ]:
def load_custom_nlp(model_name: str):
    nlp = spacy.load(model_name)
    infixes = [pattern for pattern in nlp.Defaults.infixes if "-" not in pattern]
    infix_re = compile_infix_regex(infixes)

    nlp.tokenizer = Tokenizer(
        nlp.vocab,
        prefix_search=nlp.tokenizer.prefix_search,
        suffix_search=nlp.tokenizer.suffix_search,
        infix_finditer=infix_re.finditer,
        token_match=nlp.tokenizer.token_match,
    )
    return nlp


def load_text_encoder(name_or_path: str, device: str):
    tokenizer = AutoTokenizer.from_pretrained(
        name_or_path,
        local_files_only=True,
        fix_mistral_regex=True,
        use_fast=False,
    )
    text_encoder = AutoModel.from_pretrained(
        name_or_path,
        local_files_only=True,
    )
    text_encoder.to(device)
    text_encoder.eval()
    return tokenizer, text_encoder


nlp = load_custom_nlp(spacy_model_name)
tokenizer, text_encoder = load_text_encoder(text_encoder_name_or_path, device)
print(f"Using device: {device}")

In [ ]:
file_path = "hotpot_dev_distractor_v1.json"
documents, samples = build_hotpot_retrieval_dataset(file_path, num_samples=500)

print(f"documents: {len(documents)}")
print(f"samples: {len(samples)}")
documents[0]

In [ ]:
def collect_span_embedding_records(token_embeddings, offsets, span_list, kind, document_idx, title):
    records = []
    for term, start_char, end_char in span_list:
        token_indices = get_token_indices_for_phrase(start_char, end_char, offsets)
        if not token_indices:
            continue

        embedding = token_embeddings[token_indices].mean(dim=0).to(torch.float32).numpy()
        records.append(
            {
                "kind": kind,
                "term": term,
                "document_idx": int(document_idx),
                "title": title,
                "span": (int(start_char), int(end_char)),
                "embedding": embedding,
            }
        )
    return records


def build_phrase_token_embedding_index(
    documents,
    nlp,
    text_encoder,
    tokenizer,
    device,
    batch_size=8,
    remove_duplicate_token=True,
    discard_no_word=False,
):
    cleaned_documents = [clean_text(doc["text"]) for doc in documents]

    store = {
        "documents": documents,
        "cleaned_documents": cleaned_documents,
        "index": defaultdict(list),
        "config": {
            "batch_size": batch_size,
            "remove_duplicate_token": remove_duplicate_token,
            "discard_no_word": discard_no_word,
            "encoder": text_encoder.__class__.__name__,
            "tokenizer": tokenizer.__class__.__name__,
            "device": device,
            "text_encoder_name_or_path": text_encoder_name_or_path,
            "spacy_model_name": spacy_model_name,
        },
    }

    phrase_occurrences = 0
    token_occurrences = 0

    for batch_start in range(0, len(documents), batch_size):
        batch_end = min(batch_start + batch_size, len(documents))
        batch_docs = documents[batch_start:batch_end]
        batch_texts = cleaned_documents[batch_start:batch_end]

        batch_spans = [
            extract_important_spans(
                text,
                nlp,
                min_tokens=2,
                remove_duplicate=remove_duplicate_token,
                discard_no_word=discard_no_word,
            )
            for text in batch_texts
        ]

        token_embeddings_batch, offsets_batch = encode_chunk_batch(
            batch_texts,
            text_encoder,
            tokenizer,
            device,
        )

        for local_idx, ((phrases, tokens), token_embeddings, offsets) in enumerate(
            zip(batch_spans, token_embeddings_batch, offsets_batch)
        ):
            document_idx = batch_start + local_idx
            title = batch_docs[local_idx]["title"]

            phrase_records = collect_span_embedding_records(
                token_embeddings,
                offsets,
                phrases,
                kind="phrase",
                document_idx=document_idx,
                title=title,
            )
            token_records = collect_span_embedding_records(
                token_embeddings,
                offsets,
                tokens,
                kind="token",
                document_idx=document_idx,
                title=title,
            )

            for record in phrase_records:
                store["index"][record["term"]].append(record)
            for record in token_records:
                store["index"][record["term"]].append(record)

            phrase_occurrences += len(phrase_records)
            token_occurrences += len(token_records)

        print(f"Processed {batch_end}/{len(documents)} documents")

    store["stats"] = {
        "num_documents": len(documents),
        "num_unique_terms": len(store["index"]),
        "num_phrase_occurrences": phrase_occurrences,
        "num_token_occurrences": token_occurrences,
        "num_total_occurrences": phrase_occurrences + token_occurrences,
    }
    return store


def lookup_embeddings(store, query_text, kind=None, include_text=True, include_cleaned_text=True):
    normalized_query = normalize_text(query_text.strip())
    records = list(store["index"].get(normalized_query, []))

    if kind is not None:
        records = [record for record in records if record["kind"] == kind]

    if not include_text and not include_cleaned_text:
        return records

    enriched_records = []
    for record in records:
        item = dict(record)
        document_idx = item["document_idx"]
        if include_text:
            item["text"] = store["documents"][document_idx]["text"]
        if include_cleaned_text:
            item["cleaned_text"] = store["cleaned_documents"][document_idx]
        enriched_records.append(item)
    return enriched_records


def get_document(store, document_idx):
    return store["documents"][document_idx]

In [ ]:
embedding_store = build_phrase_token_embedding_index(
    documents=documents,
    nlp=nlp,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    device=device,
    batch_size=batch_size,
    remove_duplicate_token=remove_duplicate_token,
    discard_no_word=discard_no_word,
)

with output_path.open("wb") as f:
    pickle.dump(embedding_store, f)

embedding_store["stats"]

In [ ]:
query = "United States"
records = lookup_embeddings(embedding_store, query)

print(f"query={query!r}, matches={len(records)}")

if records:
    first = records[0]
    print("kind:", first["kind"])
    print("document_idx:", first["document_idx"])
    print("title:", first["title"])
    print("span in cleaned text:", first["span"])
    print("embedding shape:", first["embedding"].shape)
    print("raw text preview:")
    print(first["text"][:500])

records[:2]